# Task 2 : Fitting a one-compartment (mono-exponential) T2 model

Compare analytical, OLS, WLS, and NLLS fitting across the preterm cohort.

In [ ]:
import sys, time, warnings
sys.path.insert(0, '../src')
warnings.filterwarnings('ignore', category=RuntimeWarning)

import numpy as np
import pandas as pd
from utils import load_preterm_cohort, load_adult_case, mean_decay_curve
from models import mono_exp, fit_two_point, fit_log_linear, fit_wls, fit_nlls
from analysis import two_point_T2_map, voxelwise_log_linear_T2, fit_cohort_mono
from plotting import (plot_two_point_maps, plot_tissue_fits_3panel,
                      plot_cohort_residuals, plot_t2_map)

## 2.1 : Load the cohorts

In [ ]:
preterm, _ = load_preterm_cohort('../data', exclude_file='../data/preterm_exclusions.csv')
case1 = load_adult_case(1)
print(f"Cohort: {len(preterm)} subjects, adult case 1: {len(case1['TE'])} echoes")

## 2.2 : Does the choice of images affect the 2-point T2?

In [ ]:
ds = preterm[0]
TE, wm = ds['TE'], mean_decay_curve(ds, 3)
print(f"All-echo log-linear T2 = {fit_log_linear(TE, wm)[1]:.1f} ms")
for j in range(1, len(TE)):
    _, t2 = fit_two_point(TE[0], wm[0], TE[j], wm[j])
    print(f"  TE pair {TE[0]:.0f}→{TE[j]:.0f} ms:  T2 = {t2:.1f} ms")

plot_two_point_maps(ds)

## 2.3 : OLS, WLS, NLLS comparison across the cohort

In [ ]:
df_methods = fit_cohort_mono(preterm)
print("Mean T2 per tissue per method:")
print(df_methods.groupby('tissue')[['T2_OLS', 'T2_WLS', 'T2_NLLS']]
      .agg(['mean', 'std']).round(1).to_string())

## 2.4 : Tissue-level fits and residuals

In [ ]:
plot_tissue_fits_3panel(preterm[0])
plot_cohort_residuals(preterm)

## 2.5 : Voxel-wise T2 map

In [ ]:
ds = preterm[0]
t0 = time.time()
T2_map = voxelwise_log_linear_T2(ds['reg'], ds['TE'], ds['mask'])
print(f"T2 map: {int(ds['mask'].sum()):,} voxels in {time.time()-t0:.1f} s")
plot_t2_map(T2_map, ds)

## 2.6 : Computational performance

In [ ]:
ds, rng = preterm[0], np.random.default_rng(0)
coords = np.argwhere(ds['mask'])
subset = coords[rng.choice(len(coords), 500, replace=False)]

def time_fn(fn):
    t0 = time.time()
    for x, y, z in subset: fn(ds['TE'], ds['reg'][x, y, z, :].astype(float))
    return time.time() - t0

t_ols = time_fn(fit_log_linear)
t_nll = time_fn(fit_nlls)
print(f"500-voxel timing: OLS = {t_ols:.3f}s, NLLS = {t_nll:.3f}s "
      f"({t_nll/max(t_ols,1e-9):.0f}× slower)")